# [JupyterHub용] HTP 요소 객체탐지 — 손그림 사전학습 → HTP 파인튜닝 (단일 파이프라인)

`baseline.ipynb` 비교 실험에서 **손그림(Quick Draw) 사전학습 → HTP**가
**COCO → HTP** 직접 파인튜닝보다 유효함을 확인했다. 이 노트북은 그 결론을 전제로,
**대조군 없이 `sketch_base.pt` → HTP 경로만** 학습한다. **Run All** 하면 아래가 순서대로 실행된다:

1. HTP 데이터 탐색 + 클래스 47개 통합
2. (캐시) Quick Draw 합성 데이터 생성 → `sketch_base.pt` 학습 — 범용 손그림 탐지 모델(**산출물 1**)
3. HTP → YOLO 변환
4. **`sketch_base.pt` → HTP 파인튜닝 → `htp_best.pt`** — HTP 전용 탐지 모델(**산출물 2**)
5. test 분할 평가 + 클래스별 지표 + 실험 기록용 요약표 + 예측 시각화

## 튜닝 방법
- 무거운 앞단계(Quick Draw 다운로드·합성·`sketch_base.pt`)는 **파일 존재 여부로 캐시**된다.
  이미 만들어져 있으면 Run All 시 즉시 통과하고 **HTP 파인튜닝만** 다시 돈다.
- 하이퍼파라미터는 **`## 2. Config` 셀** 한 곳에서 조정한다.
  자주 바꾸는 것: `MODEL_ARCH`, `IMG_SIZE`, `HTP_EPOCHS`, `SUBSET_PER_TYPE`.
  추가 인자는 `HTP_TRAIN_KW`(dict)로 Ultralytics `train()`에 그대로 전달된다(예: `lr0`, `cos_lr`, `mosaic`).

### ⚠️ 캐시 무효화 규칙 (파라미터를 바꿨을 때)
| 바꾼 것 | 지워야 반영되는 폴더/파일 |
|---|---|
| `QD_*` (합성/사전학습) | `htp_yolo/stageA_quickdraw/`, `htp_yolo/sketch_base.pt` |
| `SUBSET_PER_TYPE` | `htp_yolo/stageB_htp/` (바꾸면 반드시 삭제) |
| HTP 학습값(`HTP_EPOCHS`, `HTP_TRAIN_KW` 등) | 삭제 불필요 — 매번 재학습됨 |

### ⚠️ 실행 전 커널 재시작
Config를 고친 뒤에는 **커널 재시작 후 Run All**. 안 그러면 이전 세션의 경로/변수가 남아 산출물이 엉뚱한 곳에 저장될 수 있다.

## 1. 환경 세팅


In [ ]:
# [JupyterHub] 1) GPU 상태 확인 (드라이버 지원 CUDA 버전, GPU별 사용 현황)
!nvidia-smi

In [ ]:
# [JupyterHub] 2) 6번 GPU만 보이도록 격리 + torch 상태 확인
# 반드시 이 커널에서 torch를 처음 import하기 전에 실행해야 함
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import torch
print("torch file:", torch.__file__)
print("version:", torch.__version__, "| built for cuda:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("visible device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

In [ ]:
# [JupyterHub] 3) (필요할 때만) torch 재설치 - 드라이버가 지원하는 CUDA 12.8 이하 빌드로
# 위 셀에서 CUDA available: False 또는 RuntimeError(driver too old)가 뜨면 아래 주석을 풀고 실행,
# 그 다음 반드시 Kernel > Restart 후 2)번 셀부터 다시 실행한다.
# import sys
# print("kernel python:", sys.executable)
# !{sys.executable} -m pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# !{sys.executable} -m pip show torch   # Location이 커널 conda 환경(dodam)인지 확인

In [ ]:
# 필요 패키지 설치 (JupyterHub/VSCode 공통). 이미 설치돼 있으면 그대로 통과.
%pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pyyaml tqdm matplotlib pandas


In [ ]:
import os, sys, json, io, zipfile, random, shutil, math, urllib.request
from pathlib import Path
from urllib.parse import quote
from collections import Counter, defaultdict

import numpy as np

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    TORCH_VER = torch.__version__
except ModuleNotFoundError:
    torch = None; CUDA_AVAILABLE = False; TORCH_VER = "(미설치)"

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", TORCH_VER)
print("CUDA 사용가능:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU         :", torch.cuda.get_device_name(0), f"(x{torch.cuda.device_count()})")


In [ ]:
# [JupyterHub] 한글 폰트 등록 (Ultralytics 학습 그래프의 한글 라벨 깨짐/글리프 경고 방지)
from matplotlib import font_manager
import matplotlib.pyplot as plt
from pathlib import Path

for fp in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
           "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=fp).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

## 2. Config 셀  ⚙️

**환경을 바꿀 때는 이 셀만 수정한다.**
기본값은 스모크 테스트용이다. 본학습 값은 각 항목 주석 참고.


In [ ]:
# ═══════════════════════ Config ═══════════════════════
# 폴더/zip 이름이 바뀌어도 동작하도록 "탐색"으로 경로를 잡는다.
# (한글/영문 명명 혼재, 폴더명 변경에 견디게 하기 위함)
def _find_data_root():
    here = Path.cwd()
    for base in [here, *here.parents]:
        for cand in (base / "data", base / "ai" / "models" / "data"):
            if cand.exists():
                return cand
    return here / "data"

def _find_htp_root(data_root: Path):
    # Training/ 과 Validation/ 을 모두 가진 폴더를 하위 2단계까지 탐색
    # [JupyterHub] 이미 압축 해제된 prepared/ 결과물과 구분하기 위해
    # Training 밑에 실제 .zip 파일이 있는 폴더만 원본으로 인정한다.
    cands = [data_root]
    if data_root.exists():
        for p in data_root.iterdir():
            if p.is_dir():
                cands.append(p)
                cands.extend(q for q in p.iterdir() if q.is_dir())
    for c in cands:
        if (c / "Training").is_dir() and (c / "Validation").is_dir():
            if any((c / "Training").rglob("*.zip")):
                return c
    return None

def _find_subdir(split_dir: Path, kind: str):
    # kind='src'(원천/이미지) | 'lbl'(라벨). 한글/영문 폴더명 모두 인식
    keys = ("원천", "source", "image", "img") if kind == "src" else ("라벨", "label")
    for p in sorted(split_dir.iterdir()):
        if p.is_dir() and any(k in p.name.lower() for k in keys):
            return p
    return None

def _zip_type(stem: str):
    # 파일명에서 그림 유형 판정.
    # 주의: 'woman'이 'man'을 포함하므로 여자사람을 먼저 검사해야 한다.
    s = stem.lower()
    if "여자사람" in s or "woman" in s: return "여자사람"
    if "남자사람" in s or "man"   in s: return "남자사람"
    if "나무"     in s or "tree"  in s: return "나무"
    if "집"       in s or "house" in s: return "집"
    return None

DATA_ROOT   = _find_data_root()
HTP_ROOT    = _find_htp_root(DATA_ROOT)
# ⚠️ 절대경로 필수: 상대경로를 쓰면 Ultralytics가 project= 인자를
#    ~/AppData/Roaming/Ultralytics/settings.json 의 runs_dir(다른 프로젝트 설정값일 수 있음)와
#    합쳐 엉뚱한 위치(runs/detect/htp_yolo/runs/...)에 저장해버린다. (실제 발생한 함정)
OUTPUT_ROOT = Path("htp_yolo").resolve()         # 모든 산출물 루트
QD_RAW_DIR  = OUTPUT_ROOT / "quickdraw_raw"      # Quick Draw 원본 ndjson 캐시
QD_DS_DIR   = OUTPUT_ROOT / "stageA_quickdraw"   # Stage A 합성 데이터셋
HTP_DS_DIR  = OUTPUT_ROOT / "stageB_htp"         # HTP YOLO 데이터셋
RUNS_DIR    = OUTPUT_ROOT / "runs"

HTP_TYPES = ["집", "나무", "남자사람", "여자사람"]

# 탐색 결과 zip 레지스트리: HTP_ZIPS[split][kind][htype] -> Path
HTP_ZIPS = {}
if HTP_ROOT is not None:
    for _split in ("Training", "Validation"):
        _entry = {}
        for _kind in ("src", "lbl"):
            _sub = _find_subdir(HTP_ROOT / _split, _kind)
            _m = {}
            if _sub is not None:
                for _z in sorted(_sub.glob("*.zip")):
                    _t = _zip_type(_z.stem)
                    if _t:
                        _m[_t] = _z
            _entry[_kind] = _m
        HTP_ZIPS[_split] = _entry

# ---- 공통 학습 하이퍼파라미터 ----
MODEL_ARCH   = "yolo11n.pt"   # 스모크: yolo11n / 본학습: yolo11s (또는 yolo11m)
IMG_SIZE     = 640            # 스모크: 640 / 본학습: 960~1280 (원본 1280)
BATCH        = 16
WORKERS      = 8
DEVICE       = 0 if CUDA_AVAILABLE else "cpu"   # 오버라이드: "cpu" / 0 / "0,1"
SEED         = 42

# ---- Stage A: Quick Draw 사전학습 (sketch_base.pt 생성) ----
RUN_STAGE_A       = True    # False면 Stage A 건너뜀(단, sketch_base.pt가 이미 있어야 파인튜닝 가능)
QD_EPOCHS         = 10      # 스모크: 10 / 본학습: 30~50
QD_PER_CATEGORY   = 800     # 카테고리당 내려받을 그림 수 (스모크 800 / 본학습 3000+)
QD_TRAIN_IMAGES   = 1500    # 합성 학습 이미지 수 (스모크 1500 / 본학습 8000+)
QD_VAL_IMAGES     = 200
QD_CANVAS         = 640     # 합성 캔버스 해상도
QD_OBJ_PER_IMAGE  = (3, 9)  # 이미지당 객체 수 범위
QD_ONLY_RECOGNIZED = True   # Quick Draw가 '인식 성공'으로 표시한 그림만 사용(품질 필터)
QD_BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"

# HTP 요소와 대응되는 Quick Draw 카테고리 (존재 여부 확인 완료)
# 미존재: window, roof, chimney, smoke, hair, sneaker
QD_CATEGORIES = [
    "house", "tree", "door", "fence", "flower", "grass", "bush",
    "sun", "cloud", "moon", "star", "bird", "squirrel", "mountain",
    "face", "eye", "nose", "mouth", "ear", "arm", "hand", "leg", "foot",
    "shoe", "sock", "t-shirt", "pants", "swing set", "ladder", "stairs",
    "river", "pool", "garden",
]

# ---- HTP 파인튜닝 ----
HTP_EPOCHS      = 5      # 스모크: 5 / 본학습: 50~100
PATIENCE        = 20
SUBSET_PER_TYPE = 200    # 스모크: 유형별 200장. None이면 전체(유형별 11,200장)
TEST_FRACTION   = 0.10   # 공식 Training에서 test로 분리할 비율 (val은 공식 Validation 사용)

# ---- 추가 튜닝: Ultralytics train() 인자를 자유롭게 전달 ----
# 예) {"lr0": 0.001, "cos_lr": True, "mosaic": 0.5, "optimizer": "AdamW", "close_mosaic": 10}
HTP_TRAIN_KW = {}

random.seed(SEED); np.random.seed(SEED)
for d in (OUTPUT_ROOT, QD_RAW_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT    :", DATA_ROOT)
print("HTP_ROOT     :", HTP_ROOT)
if HTP_ROOT is None:
    print("  ⚠️ Training/Validation 폴더를 찾지 못했습니다. DATA_ROOT를 직접 지정하세요.")
else:
    for _s in HTP_ZIPS:
        _miss = [t for t in HTP_TYPES
                 if t not in HTP_ZIPS[_s]["src"] or t not in HTP_ZIPS[_s]["lbl"]]
        print(f"  {_s}: 원천 {len(HTP_ZIPS[_s]['src'])}/4, 라벨 {len(HTP_ZIPS[_s]['lbl'])}/4"
              + (f"  ⚠️ 누락: {_miss}" if _miss else "  ✓"))
print("OUTPUT_ROOT  :", OUTPUT_ROOT)
print(f"arch={MODEL_ARCH} imgsz={IMG_SIZE} batch={BATCH} device={DEVICE}")
print(f"Stage A 실행={RUN_STAGE_A} (epochs={QD_EPOCHS}, 합성 {QD_TRAIN_IMAGES}장, 카테고리 {len(QD_CATEGORIES)}개)")
print(f"HTP 파인튜닝 epochs={HTP_EPOCHS}, SUBSET_PER_TYPE={SUBSET_PER_TYPE} (None=전체)")
print(f"HTP_TRAIN_KW={HTP_TRAIN_KW}")

## 3. 데이터 탐색 (HTP)

### 파악된 HTP 어노테이션 포맷 (JSON)
```
{
  "meta": { "img_resolution": "1280x1280", "age": 10, "sex": "남", ... },
  "annotations": {
    "class": "집",
    "bbox": [ {"label": "집전체", "x": 220, "y": 222, "w": 402, "h": 529}, ... ]
  }
}
```
- `x, y` = bbox **좌상단** 픽셀 좌표, `w, h` = 픽셀 폭/높이. 모든 이미지 **1280×1280**.
- 라벨 파일명 = 이미지 파일명(확장자만 다름): `집_10_남_00137.json` ↔ `집_10_남_00137.jpg`.
- 검증 결과: 이미지 50,400장 / 라벨 50,400개 **완전 1:1 매칭, 미매칭 0**.


In [ ]:
# ── HTP zip 접근 헬퍼 ──────────────────────────────────────────────
def fix_name(name: str) -> str:
    # zip 내부 파일명 유니코드 복원. 환경에 따라 cp437로 깨진 경우만 보정.
    if "�" in name:
        try:
            return name.encode("cp437").decode("utf-8")
        except Exception:
            pass
    return name

def htp_zip_path(split: str, kind: str, htype: str):
    # Config에서 탐색해 둔 레지스트리 조회. 없으면 None.
    return HTP_ZIPS.get(split, {}).get(kind, {}).get(htype)

def zip_members(zip_path: Path, ext: str) -> dict:
    # stem -> zip 내부 원본 멤버명
    out = {}
    with zipfile.ZipFile(zip_path) as z:
        for n in z.namelist():
            if n.lower().endswith(ext):
                out[Path(fix_name(n)).stem] = n
    return out

def read_label_json(zip_path: Path, member: str) -> dict:
    with zipfile.ZipFile(zip_path) as z:
        return json.loads(z.read(member))

def read_image_bytes(zip_path: Path, member: str) -> bytes:
    with zipfile.ZipFile(zip_path) as z:
        return z.read(member)

print("헬퍼 정의 완료.")


In [ ]:
# ── 데이터셋 인벤토리 ─────────────────────────────────────────────
import pandas as pd
rows = []
for split in HTP_ZIPS:
    for htype in HTP_TYPES:
        sp, lp = htp_zip_path(split, "src", htype), htp_zip_path(split, "lbl", htype)
        n_img = len(zip_members(sp, ".jpg")) if sp else 0
        n_lbl = len(zip_members(lp, ".json")) if lp else 0
        rows.append({"split": split, "유형": htype, "이미지": n_img, "라벨": n_lbl,
                     "zip": (sp.name if sp else "없음") + " / " + (lp.name if lp else "없음")})
inv = pd.DataFrame(rows)
display(inv)
print("총 이미지:", inv["이미지"].sum(), "/ 총 라벨:", inv["라벨"].sum())


In [ ]:
# ── 샘플 이미지 + bbox 시각화 ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageChops

# 한글 폰트(있으면 사용) — 없으면 라벨 텍스트가 깨져 보일 수 있음
for fp in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
           "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=fp).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

def show_htp_sample(split, htype, n=2):
    sp, lp = htp_zip_path(split, "src", htype), htp_zip_path(split, "lbl", htype)
    imgs, lbls = zip_members(sp, ".jpg"), zip_members(lp, ".json")
    stems = [s for s in sorted(lbls) if s in imgs][:n]
    fig, axes = plt.subplots(1, len(stems), figsize=(6 * len(stems), 6))
    if len(stems) == 1:
        axes = [axes]
    for ax, stem in zip(axes, stems):
        im = Image.open(io.BytesIO(read_image_bytes(sp, imgs[stem]))).convert("RGB")
        ax.imshow(im); ax.set_title(f"{htype} / {stem}", fontsize=9); ax.axis("off")
        for b in read_label_json(lp, lbls[stem])["annotations"]["bbox"]:
            ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"],
                                           fill=False, edgecolor="red", linewidth=1.2))
            ax.text(b["x"], max(b["y"] - 3, 0), b["label"], color="white", fontsize=7,
                    bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
    plt.tight_layout(); plt.show()

show_htp_sample("Training", "집", n=2)
show_htp_sample("Training", "남자사람", n=1)


> **참고 — 사람 그림의 신발 라벨**: 모든 사람 그림에 `운동화` 2개와 `남자구두`/`여자구두` 2개가
> **동시에** 존재한다. 확인 결과 이는 오류가 아니라, 아이가 인물을 그린 뒤 **신발을 따로 확대해서
> 그리는** HTP 검사 절차 때문이다(인물 영역 밖에 신발 그림이 별도로 존재). 중복 제거 불필요.


## 4. 클래스 체계 통합  🏷️

HTP 라벨을 **데이터에서 직접 수집**해 통합 클래스 목록을 만든다(하드코딩 금지).

- 4개 유형에 걸쳐 **총 47개 라벨**. 동일 라벨 문자열은 유형 간 자동 통합된다.
  - `꽃`(집·나무 공통), `나무`(집 그림 배경) → 각 1개 클래스
  - 사람 부위(`눈`·`팔`…)는 남녀 라벨이 동일 → 성별 무관 단일 클래스
  - 성별 구분 라벨만 별도 유지: `남자구두`·`여자구두` (`운동화`는 공통)
- 클래스 id는 **가나다 정렬로 결정론적** 부여 → 재실행 시 항상 동일.


In [ ]:
# ── 전체 라벨 수집 (Training 라벨 zip 전수 스캔, 약 1~2분) ────────
from tqdm.auto import tqdm

label_counter = Counter()
label_by_type = defaultdict(Counter)

for htype in HTP_TYPES:
    lp = htp_zip_path("Training", "lbl", htype)
    members = zip_members(lp, ".json")
    with zipfile.ZipFile(lp) as z:
        for stem, m in tqdm(members.items(), desc=f"scan {htype}", leave=False):
            for b in json.loads(z.read(m))["annotations"]["bbox"]:
                label_counter[b["label"]] += 1
                label_by_type[htype][b["label"]] += 1

CLASS_NAMES = sorted(label_counter)
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_ROOT / "classes.json", "w", encoding="utf-8") as f:
    json.dump({"names": CLASS_NAMES, "class_to_id": CLASS_TO_ID}, f, ensure_ascii=False, indent=2)

print(f"통합 클래스 수: {len(CLASS_NAMES)}  (총 bbox {sum(label_counter.values()):,}개)")
rows = [{"id": CLASS_TO_ID[n], "클래스": n, "bbox수(train)": label_counter[n],
         "등장유형": ", ".join(t for t in HTP_TYPES if n in label_by_type[t])}
        for n in CLASS_NAMES]
pd.set_option("display.max_rows", None)
display(pd.DataFrame(rows))


### Quick Draw ↔ HTP 클래스 대응

Stage A(Quick Draw)와 Stage B(HTP)는 **클래스 수가 다르므로 검출 헤드는 재초기화**된다.
전이되는 가치는 **백본/넥이 학습한 선화(line-art) 특징**이며, 이는 의도된 설계다.
아래 대응표는 "Stage A가 HTP에 얼마나 관련된 것을 배우는가"를 보여주는 참고자료다.


In [ ]:
# ── Quick Draw → HTP 대응 참고표 ──────────────────────────────────
QD_TO_HTP = {
    "house": "집전체", "tree": "나무전체", "door": "문", "fence": "울타리",
    "flower": "꽃", "grass": "잔디", "bush": "잔디(유사)", "sun": "태양",
    "cloud": "구름", "moon": "달", "star": "별", "bird": "새",
    "squirrel": "다람쥐", "mountain": "산", "face": "얼굴", "eye": "눈",
    "nose": "코", "mouth": "입", "ear": "귀", "arm": "팔", "hand": "손",
    "leg": "다리", "foot": "발", "shoe": "남자구두/여자구두", "sock": "발(유사)",
    "t-shirt": "상체", "pants": "다리(유사)", "swing set": "그네",
    "pool": "연못", "river": "길(유사)", "ladder": "(보조)",
    "stairs": "(보조)", "garden": "(보조)",
}
qd_rows = [{"QuickDraw": c, "HTP 대응": QD_TO_HTP.get(c, "(보조)")} for c in QD_CATEGORIES]
display(pd.DataFrame(qd_rows))

covered = {v for v in QD_TO_HTP.values() if v in CLASS_NAMES}
print(f"HTP 47개 중 Quick Draw로 직접 대응되는 클래스: {len(covered)}개")
print("대응 없는 HTP 클래스(예):",
      ", ".join(sorted(set(CLASS_NAMES) - covered)[:15]), "...")


## 5. Stage A — Quick Draw 합성 데이터셋 생성  🎨

### 핵심 아이디어
Quick Draw 원본은 **한 그림에 객체 1개**이고 좌표가 0–255를 꽉 채우도록 정규화되어 있어,
그대로 쓰면 bbox가 항상 이미지 전체가 된다(퇴화 과제). 따라서:

1. 카테고리별 스케치를 스트리밍으로 내려받는다(필요한 만큼만 읽고 조기 종료 → 전체 48MB+ 다운로드 불필요).
2. 스케치를 **랜덤 크기로 렌더링**해 빈 캔버스의 **랜덤 위치에 합성**한다.
3. 배치를 우리가 하므로 **bbox를 정확히 계산**할 수 있다(스트로크 좌표 min/max + 선 두께 보정).
4. 연필 느낌에 가깝도록 선 두께·농도·약간의 회전을 무작위화해 도메인 격차를 줄인다.


In [ ]:
# ── Quick Draw 다운로드 (필요한 줄만 스트리밍, 캐시) ──────────────
def qd_download(category: str, n: int, dest_dir: Path) -> Path:
    dest = dest_dir / f"{category.replace(' ', '_')}.ndjson"
    if dest.exists():
        with dest.open(encoding="utf-8") as f:
            if sum(1 for _ in f) >= n:
                return dest   # 캐시 충분
    url = QD_BASE_URL + quote(category) + ".ndjson"
    lines = []
    req = urllib.request.Request(url, headers={"User-Agent": "htp-baseline/1.0"})
    with urllib.request.urlopen(req, timeout=180) as r:
        for raw in r:                      # ndjson: 줄 단위 스트리밍
            lines.append(raw.decode("utf-8"))
            if len(lines) >= n:
                break                      # 필요한 만큼만 받고 조기 종료
    dest.write_text("".join(lines), encoding="utf-8")
    return dest

def qd_load(category: str, n: int) -> list:
    # 스트로크 리스트 반환. 각 스트로크는 [[x...], [y...]] (0~255 좌표계)
    p = QD_RAW_DIR / f"{category.replace(' ', '_')}.ndjson"
    out = []
    with p.open(encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            if QD_ONLY_RECOGNIZED and not d.get("recognized", True):
                continue
            out.append(d["drawing"])
            if len(out) >= n:
                break
    return out

QD_POOL = {}
for cat in tqdm(QD_CATEGORIES, desc="Quick Draw 다운로드"):
    qd_download(cat, QD_PER_CATEGORY, QD_RAW_DIR)
    QD_POOL[cat] = qd_load(cat, QD_PER_CATEGORY)

QD_NAMES   = sorted(QD_POOL)                       # Stage A 클래스(결정론적 정렬)
QD_NAME_ID = {c: i for i, c in enumerate(QD_NAMES)}
print(f"\n카테고리 {len(QD_NAMES)}개, 그림 총 {sum(len(v) for v in QD_POOL.values()):,}개 확보")
print("카테고리별 개수 예시:", {c: len(QD_POOL[c]) for c in QD_NAMES[:5]})


In [ ]:
# ── 스케치 렌더링 + 합성 캔버스 생성 ──────────────────────────────
def render_sketch(strokes, size: int, rng: random.Random):
    # 스케치를 size x size 패치(L 모드, 흰 배경)에 렌더링.
    # 반환: (패치, 패치 내 tight bbox (x1,y1,x2,y2))
    pts = [(x, y) for s in strokes for x, y in zip(s[0], s[1])]
    if len(pts) < 2:
        return None, None
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    x0, x1 = min(xs), max(xs); y0, y1 = min(ys), max(ys)
    sw = max(1, int(round(size / rng.choice([70, 90, 120]))))   # 선 두께 무작위
    pad = sw + 2
    span = max(x1 - x0, y1 - y0, 1)
    scale = (size - 2 * pad) / span

    patch = Image.new("L", (size, size), 255)
    dr = ImageDraw.Draw(patch)
    ink = rng.randint(40, 110)          # 연필 농도 무작위
    def tx(x, y):
        return (pad + (x - x0) * scale, pad + (y - y0) * scale)
    for s in strokes:
        pl = [tx(x, y) for x, y in zip(s[0], s[1])]
        if len(pl) == 1:
            dr.ellipse([pl[0][0]-sw, pl[0][1]-sw, pl[0][0]+sw, pl[0][1]+sw], fill=ink)
        else:
            dr.line(pl, fill=ink, width=sw, joint="curve")
    if rng.random() < 0.5:              # 약간의 회전
        patch = patch.rotate(rng.uniform(-8, 8), resample=Image.BILINEAR, fillcolor=255)

    # tight bbox는 렌더링이 모두 끝난 뒤 '실제 잉크 픽셀'에서 계산한다.
    # (좌표 계산으로 구하면 회전이 적용됐을 때 잉크가 박스 밖으로 삐져나간다)
    arr = np.asarray(patch)
    ink_px = np.argwhere(arr < 250)     # 배경 255, 안티에일리어싱 여유
    if ink_px.size == 0:
        return None, None
    (iy1, ix1), (iy2, ix2) = ink_px.min(0), ink_px.max(0)
    bb = (float(ix1), float(iy1), float(ix2 + 1), float(iy2 + 1))
    return patch, bb

def make_canvas(rng: random.Random):
    # Quick Draw 스케치를 랜덤 배치한 합성 장면 1장 생성.
    C = QD_CANVAS
    canvas = Image.new("L", (C, C), rng.randint(248, 255))   # 살짝 종이톤
    boxes = []
    for _ in range(rng.randint(*QD_OBJ_PER_IMAGE)):
        cat = rng.choice(QD_NAMES)
        if not QD_POOL[cat]:
            continue
        strokes = rng.choice(QD_POOL[cat])
        s = rng.randint(int(C * 0.12), int(C * 0.42))
        patch, bb = render_sketch(strokes, s, rng)
        if patch is None:
            continue
        px, py = rng.randint(0, C - s), rng.randint(0, C - s)
        region = canvas.crop((px, py, px + s, py + s))
        canvas.paste(ImageChops.darker(region, patch), (px, py))  # 겹침 시 진한 쪽 유지
        x1, y1, x2, y2 = bb[0] + px, bb[1] + py, bb[2] + px, bb[3] + py
        if x2 - x1 >= 4 and y2 - y1 >= 4:
            boxes.append((QD_NAME_ID[cat], x1, y1, x2, y2))
    return canvas.convert("RGB"), boxes

def write_yolo_split(out_dir: Path, n_images: int, seed: int, desc: str):
    img_d, lbl_d = out_dir / "images", out_dir / "labels"
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    C = QD_CANVAS
    for i in tqdm(range(n_images), desc=desc, leave=False):
        img, boxes = make_canvas(rng)
        img.save(img_d / f"qd_{i:06d}.jpg", quality=92)
        lines = [f"{c} {((x1+x2)/2)/C:.6f} {((y1+y2)/2)/C:.6f} {(x2-x1)/C:.6f} {(y2-y1)/C:.6f}"
                 for c, x1, y1, x2, y2 in boxes]
        (lbl_d / f"qd_{i:06d}.txt").write_text("\n".join(lines), encoding="utf-8")

if RUN_STAGE_A:
    if (QD_DS_DIR / "train" / "images").exists() and \
       len(list((QD_DS_DIR / "train" / "images").glob("*.jpg"))) >= QD_TRAIN_IMAGES:
        print("Stage A 합성 데이터셋 캐시 사용:", QD_DS_DIR.resolve())
    else:
        shutil.rmtree(QD_DS_DIR, ignore_errors=True)
        write_yolo_split(QD_DS_DIR / "train", QD_TRAIN_IMAGES, SEED,     "합성 train")
        write_yolo_split(QD_DS_DIR / "val",   QD_VAL_IMAGES,   SEED + 1, "합성 val")
        print("생성 완료:", QD_DS_DIR.resolve())
    qd_yaml = QD_DS_DIR / "data.yaml"
    qd_yaml.write_text(
        "path: " + str(QD_DS_DIR.resolve()) + "\ntrain: train/images\nval: val/images\n"
        + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(QD_NAMES)),
        encoding="utf-8")
    print("data.yaml:", qd_yaml.resolve(), "| 클래스", len(QD_NAMES), "개")
else:
    print("RUN_STAGE_A=False → Stage A 건너뜀")


In [ ]:
# ── 합성 결과 검증: bbox를 이미지 위에 그려 확인 ──────────────────
if RUN_STAGE_A:
    smp = sorted((QD_DS_DIR / "train" / "images").glob("*.jpg"))[:3]
    fig, axes = plt.subplots(1, len(smp), figsize=(6 * len(smp), 6))
    if len(smp) == 1: axes = [axes]
    for ax, ip in zip(axes, smp):
        im = Image.open(ip); W, H = im.size
        ax.imshow(im, cmap="gray"); ax.axis("off"); ax.set_title(ip.name, fontsize=9)
        for line in (ip.parent.parent / "labels" / (ip.stem + ".txt")).read_text().splitlines():
            c, xc, yc, w, h = line.split()
            c = int(c); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
            ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                           fill=False, edgecolor="red", linewidth=1.2))
            ax.text(xc-w/2, max(yc-h/2-3, 0), QD_NAMES[c], color="white", fontsize=7,
                    bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
    plt.tight_layout(); plt.show()


## 6. Stage A 학습 — 범용 손그림 탐지기

COCO 사전학습 가중치에서 시작해 합성 Quick Draw 장면으로 학습한다.
산출물 `sketch_base.pt`가 Stage B의 시작점이 된다.


In [ ]:
from ultralytics import YOLO

SKETCH_BASE = OUTPUT_ROOT / "sketch_base.pt"

if RUN_STAGE_A:
    if SKETCH_BASE.exists():
        print("기존 sketch_base.pt 재사용:", SKETCH_BASE.resolve())
    else:
        m = YOLO(MODEL_ARCH)                      # COCO 사전학습에서 시작
        m.train(data=str((QD_DS_DIR / "data.yaml").resolve()),
                epochs=QD_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, workers=WORKERS,
                device=DEVICE, seed=SEED, project=str(RUNS_DIR), name="stageA_quickdraw",
                exist_ok=True, verbose=True)
        best = RUNS_DIR / "stageA_quickdraw" / "weights" / "best.pt"
        shutil.copy(best, SKETCH_BASE)
        print("Stage A 완료 →", SKETCH_BASE.resolve())
else:
    print("RUN_STAGE_A=False → Stage A 학습 건너뜀")


## 7. HTP → YOLO 어노테이션 변환

`{label, x, y, w, h}` (좌상단·픽셀) → `class x_center y_center w h` (정규화).
- 좌표는 이미지 경계로 **클리핑**하고, 폭/높이가 0 이하가 되는 박스는 **버린다**.
- 원본 zip은 **읽기 전용**으로만 접근하고, 결과는 `htp_yolo/stageB_htp/` 에만 생성한다.


In [ ]:
# ── 변환 함수 ─────────────────────────────────────────────────────
def parse_resolution(meta) -> tuple:
    try:
        w, h = str(meta.get("img_resolution", "1280x1280")).lower().split("x")
        return int(w), int(h)
    except Exception:
        return 1280, 1280

def bbox_to_yolo(b, W, H):
    # 좌상단(x,y,w,h) -> 정규화 (cls_id, xc, yc, w, h). 유효하지 않으면 None
    cid = CLASS_TO_ID.get(b["label"])
    if cid is None:
        return None
    x1 = max(0.0, float(b["x"]));            y1 = max(0.0, float(b["y"]))
    x2 = min(float(W), x1 + float(b["w"]));  y2 = min(float(H), y1 + float(b["h"]))
    if x2 - x1 <= 1 or y2 - y1 <= 1:
        return None
    return cid, ((x1+x2)/2)/W, ((y1+y2)/2)/H, (x2-x1)/W, (y2-y1)/H

def select_stems(split, htype, limit):
    sp, lp = htp_zip_path(split, "src", htype), htp_zip_path(split, "lbl", htype)
    common = sorted(set(zip_members(sp, ".jpg")) & set(zip_members(lp, ".json")))
    random.Random(SEED).shuffle(common)
    return common[:limit] if limit else common

def convert(split, htype, stems, out_dir):
    # zip에서 이미지 추출 + YOLO txt 생성
    sp, lp = htp_zip_path(split, "src", htype), htp_zip_path(split, "lbl", htype)
    imgs, lbls = zip_members(sp, ".jpg"), zip_members(lp, ".json")
    img_d, lbl_d = out_dir / "images", out_dir / "labels"
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    n_box = n_drop = 0
    with zipfile.ZipFile(sp) as zs, zipfile.ZipFile(lp) as zl:
        for stem in tqdm(stems, desc=f"{out_dir.name}/{htype}", leave=False):
            (img_d / f"{stem}.jpg").write_bytes(zs.read(imgs[stem]))
            d = json.loads(zl.read(lbls[stem]))
            W, H = parse_resolution(d["meta"])
            lines = []
            for b in d["annotations"]["bbox"]:
                r = bbox_to_yolo(b, W, H)
                if r is None:
                    n_drop += 1; continue
                lines.append("%d %.6f %.6f %.6f %.6f" % r); n_box += 1
            (lbl_d / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    return n_box, n_drop

print("변환 함수 정의 완료.")


In [ ]:
# ── 실행: 공식 분할 기반 train / val / test 구성 ──────────────────
# Training zip -> train + test(TEST_FRACTION),  Validation zip -> val
if HTP_DS_DIR.exists() and (HTP_DS_DIR / "train" / "images").exists():
    print("기존 변환 결과 재사용:", HTP_DS_DIR.resolve(), "(재생성하려면 폴더 삭제)")
else:
    tot_box = tot_drop = 0
    for htype in HTP_TYPES:
        tr = select_stems("Training", htype, SUBSET_PER_TYPE)
        k = max(1, int(len(tr) * TEST_FRACTION))
        test_stems, train_stems = tr[:k], tr[k:]
        va = select_stems("Validation", htype,
                          max(1, int(SUBSET_PER_TYPE * 0.25)) if SUBSET_PER_TYPE else None)
        for stems, name, split in ((train_stems, "train", "Training"),
                                   (test_stems,  "test",  "Training"),
                                   (va,          "val",   "Validation")):
            b, d = convert(split, htype, stems, HTP_DS_DIR / name)
            tot_box += b; tot_drop += d
    print(f"변환 완료 — 유효 bbox {tot_box:,}개, 제외된 bbox {tot_drop:,}개")

for name in ("train", "val", "test"):
    n = len(list((HTP_DS_DIR / name / "images").glob("*.jpg")))
    print(f"  {name:6s}: {n:,}장")


In [ ]:
# ── 변환 검증: YOLO txt를 되읽어 bbox를 그려 확인 ─────────────────
ID_TO_CLASS = {i: n for n, i in CLASS_TO_ID.items()}
smp = sorted((HTP_DS_DIR / "train" / "images").glob("*.jpg"))[:3]
fig, axes = plt.subplots(1, len(smp), figsize=(6 * len(smp), 6))
if len(smp) == 1: axes = [axes]
for ax, ip in zip(axes, smp):
    im = Image.open(ip); W, H = im.size
    ax.imshow(im); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    for line in (ip.parent.parent / "labels" / (ip.stem + ".txt")).read_text().splitlines():
        c, xc, yc, w, h = line.split()
        c = int(c); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                       fill=False, edgecolor="lime", linewidth=1.2))
        ax.text(xc-w/2, max(yc-h/2-3, 0), ID_TO_CLASS[c], color="black", fontsize=7,
                bbox=dict(facecolor="lime", alpha=0.7, pad=0.5, edgecolor="none"))
plt.tight_layout(); plt.show()
print("↑ 녹색 박스가 원본 라벨과 일치하면 변환이 올바른 것이다.")


In [ ]:
# ── data.yaml 생성 (Stage B) ──────────────────────────────────────
htp_yaml = HTP_DS_DIR / "data.yaml"
htp_yaml.write_text(
    "path: " + str(HTP_DS_DIR.resolve()) +
    "\ntrain: train/images\nval: val/images\ntest: test/images\n"
    + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)),
    encoding="utf-8")
print(htp_yaml.read_text(encoding="utf-8")[:400], "...")
print("\n클래스 수:", len(CLASS_NAMES))


## 8. HTP 파인튜닝 학습 (sketch_base → HTP)

`sketch_base.pt`(범용 손그림 탐지 모델)를 시작 가중치로 HTP 47클래스에 파인튜닝한다.
Stage A와 HTP는 클래스 수가 달라(33 vs 47) **검출 헤드는 자동 재초기화**되고, 백본/넥의 선화 특징이 전이된다.
결과는 `htp_best.pt`로 저장된다. (대조군 COCO 경로는 제거 — `baseline.ipynb`에서 이미 열세 확인)

In [ ]:
def train_htp(init_weights, run_name):
    m = YOLO(str(init_weights))
    m.train(data=str(htp_yaml.resolve()),
            epochs=HTP_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, workers=WORKERS,
            device=DEVICE, seed=SEED, patience=PATIENCE,
            project=str(RUNS_DIR), name=run_name, exist_ok=True, verbose=True,
            **HTP_TRAIN_KW)
    return RUNS_DIR / run_name / "weights" / "best.pt"

# 범용 손그림 모델(sketch_base.pt)에서 HTP로 파인튜닝하는 단일 경로.
# sketch_base.pt가 없으면 섹션 5~6(Stage A)을 먼저 실행해야 한다
# (RUN_STAGE_A=True로 두고 Run All 하면 자동 생성 → 이 셀이 그 결과를 이어받는다).
if not SKETCH_BASE.exists():
    raise FileNotFoundError(
        f"sketch_base.pt가 없습니다: {SKETCH_BASE}\n"
        "→ 섹션 5~6(Stage A)을 먼저 실행하세요. RUN_STAGE_A=True면 Run All 시 자동 생성됩니다."
    )

RUN_NAME = "htp_from_sketch"
best_pt  = train_htp(SKETCH_BASE, RUN_NAME)

# 최종 산출물로 복사 (HTP 전용 탐지 모델)
HTP_BEST = OUTPUT_ROOT / "htp_best.pt"
shutil.copy(best_pt, HTP_BEST)
print("\nHTP 파인튜닝 완료 → 최종 모델:", HTP_BEST.resolve())

## 9. 평가 / 시각화

`test` 분할(공식 Training에서 분리, 학습에 미사용)로 평가한다.
mAP@50 · mAP@50-95 · 클래스별 precision/recall을 출력하고 예측 결과를 시각화한다.


In [ ]:
def evaluate(weights, tag):
    m = YOLO(str(weights))
    r = m.val(data=str(htp_yaml.resolve()), split="test", imgsz=IMG_SIZE,
              batch=BATCH, device=DEVICE, project=str(RUNS_DIR),
              name=f"eval_{tag}", exist_ok=True, verbose=False)
    return {"mAP@50": r.box.map50, "mAP@50-95": r.box.map,
            "precision": r.box.mp, "recall": r.box.mr}, r

metric, raw_result = evaluate(HTP_BEST, "htp")

summary = pd.DataFrame({"htp_from_sketch": metric}).T
display(summary.style.format("{:.4f}"))

In [ ]:
# ── 클래스별 precision / recall ───────────────────────────────────
r = raw_result
rows = []
for i, c in enumerate(r.box.ap_class_index):
    rows.append({"클래스": CLASS_NAMES[c],
                 "precision": r.box.p[i], "recall": r.box.r[i],
                 "mAP@50": r.box.ap50[i], "mAP@50-95": r.box.ap[i]})
per_cls = pd.DataFrame(rows).sort_values("mAP@50-95", ascending=False)
display(per_cls.style.format({c: "{:.4f}" for c in per_cls.columns if c != "클래스"}))
print("\n성능 하위 10개 클래스:")
display(per_cls.tail(10))

In [ ]:
# ── 실험 기록용 요약 표 (수동 기록 편의) ─────────────────────────
# 이 노트북은 sketch_base.pt(고정·캐시)에서 HTP 파인튜닝만 튜닝하므로,
# 기록 항목도 HTP 파인튜닝 파라미터 + 성능 지표로 한정한다.
# 파라미터를 바꿔 재실행할 때마다 이 셀을 실행 → TSV 한 줄을 스프레드시트에 붙여넣으면 실험 로그가 된다.
import datetime

def _htp_train_sec():
    p = RUNS_DIR / RUN_NAME / "results.csv"
    if not p.exists():
        return None
    return round(float(pd.read_csv(p).iloc[-1]["time"]), 1)

def _worst_classes(r, n=3):
    order = np.argsort(r.box.ap)[:n]
    return "; ".join(f"{CLASS_NAMES[r.box.ap_class_index[i]]}={r.box.ap[i]:.3f}" for i in order)

row = {
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "init": "sketch_base",          # 시작 가중치(고정)
    "arch": MODEL_ARCH, "imgsz": IMG_SIZE, "batch": BATCH,
    "subset_per_type": SUBSET_PER_TYPE if SUBSET_PER_TYPE else "전체",
    "htp_epochs": HTP_EPOCHS, "patience": PATIENCE,
    "train_kw": str(HTP_TRAIN_KW) if HTP_TRAIN_KW else "-",
    "train_sec": _htp_train_sec(),
    "mAP50": round(metric["mAP@50"], 4),
    "mAP50-95": round(metric["mAP@50-95"], 4),
    "precision": round(metric["precision"], 4),
    "recall": round(metric["recall"], 4),
    "worst3": _worst_classes(raw_result),
}
log_row = pd.DataFrame([row])

print("=== 이번 실험 요약 (sketch_base → HTP) ===")
display(log_row.T.rename(columns={0: "값"}))

print("\n=== 스프레드시트 붙여넣기용 (헤더 행 → 값 행, 탭 구분) ===")
print("\t".join(log_row.columns))
print("\t".join(str(v) for v in log_row.iloc[0]))

In [ ]:
# ── 예측 결과 시각화 (test 분할) ──────────────────────────────────
model = YOLO(str(HTP_BEST))
test_imgs = sorted((HTP_DS_DIR / "test" / "images").glob("*.jpg"))[:4]
preds = model.predict([str(p) for p in test_imgs], imgsz=IMG_SIZE, conf=0.25,
                      device=DEVICE, verbose=False)

pred_dir = OUTPUT_ROOT / "predictions"; pred_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6))
if len(preds) == 1: axes = [axes]
for ax, p, ip in zip(axes, preds, test_imgs):
    im = p.plot()[:, :, ::-1]           # BGR -> RGB
    ax.imshow(im); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    Image.fromarray(im).save(pred_dir / f"pred_{ip.stem}.jpg")
plt.tight_layout(); plt.show()
print("예측 이미지 저장:", pred_dir.resolve())

## 정리

### 산출물 (모두 `htp_yolo/` 아래)
| 경로 | 내용 |
|---|---|
| `classes.json` | HTP 47개 클래스 ↔ id 매핑 |
| `sketch_base.pt` | **범용 손그림 탐지 모델 (산출물 1)** |
| `htp_best.pt` | **HTP 전용 탐지 모델 — sketch→HTP 파인튜닝 최종본 (산출물 2)** |
| `runs/htp_from_sketch/` | 학습 로그·곡선·`weights/best.pt` |
| `runs/eval_htp/` | test 분할 평가 결과 |
| `predictions/` | 예측 시각화 이미지 |

### 반복 실험 워크플로
1. Config에서 파라미터 수정 → (필요 시 상단 "캐시 무효화 규칙"대로 폴더 삭제)
2. **커널 재시작 → Run All**
3. 요약표 셀의 TSV 한 줄을 스프레드시트에 추가 기록
   (`runs/htp_from_sketch/`는 `exist_ok=True`라 재실행 시 덮어써지므로 지표는 매번 기록해 둘 것)

### 본학습 전환 예시 (Config)
```python
MODEL_ARCH      = "yolo11s.pt"
IMG_SIZE        = 960
SUBSET_PER_TYPE = None      # 전체 44,800장
HTP_EPOCHS      = 50
HTP_TRAIN_KW    = {"cos_lr": True}
```